In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [67]:
# Basic MHA implementations for self attention
class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_in should be integer multiples of n_heads!"
        self.d_model = d_model # i.e., d_model, or d_embed
        self.d_head = d_model // n_heads
        self.n_heads = n_heads
        self.scale = self.d_head**-0.5 # 1/sqrt(d_k)

        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x, is_causal=False):
        # dim (batch, seq_len, d_model)
        batch_size, seq_len, _ = x.shape

        # dim (batch, n_heads, seq_len, d_head)
        k = self.W_k(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1,2).contiguous()
        v = self.W_v(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1,2).contiguous()
        q = self.W_q(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1,2).contiguous()

        # dim (batch, n_heads, seq_len, seq_len)
        attn = q @ k.transpose(-2, -1) * self.scale
        
        if is_causal:
            causal_mask = torch.triu(torch.ones_like(attn, device=x.device, dtype=torch.bool), diagonal=1)
            attn.masked_fill(causal_mask, -torch.inf)

        norm_attn = torch.softmax(attn, dim=-1)
        # dim (batch, n_heads, seq_len, d_head)
        output = (attn @ v).transpose(1,2).reshape(batch_size, seq_len, self.d_model).contiguous()

        output = self.out_proj(output)
        
        return output

In [68]:
x_numpy = np.random.randn(1, 25, 48)
x = torch.from_numpy(x_numpy).float()

In [69]:
sa = SelfAttention(48, 8)

In [70]:
output = sa(x, is_causal=True)

In [71]:
output.size()

torch.Size([1, 25, 48])

In [114]:
# Task use transformer for sentence sentiment classification
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=2048, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.mha = SelfAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.proj = nn.Linear(d_model, 2)
        
    def forward(self, x):
        # x dim: (batch_size, seq_len, d_model)
        x = self.norm1(x)
        x = self.mha(x)
        x = self.norm2(x)
        x = self.ff(x)
        
        # dim: (batch_size, seq_len, 2)
        x = self.proj(x)

        # mean pooling, dim (batch_size, 2)
        x = torch.mean(x, dim=1)
        
        return x

In [192]:
# build a training loop
num_sample = 500
batch_size = 32
learning_rate = 1e-3
embedding_dim = 48
seq_len = 25
num_epochs = 100

In [193]:
x = torch.rand((num_sample, seq_len, embedding_dim))
y = torch.randint(0, 2, (num_sample,)).long()

In [194]:
classifier = TransformerBlock(embedding_dim, 8)

In [195]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=learning_rate)

In [196]:
dataset = torch.utils.data.TensorDataset(x, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size)

In [197]:
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in dataloader:
        optimizer.zero_grad()
        outputs = classifier(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'{epoch+1}/{num_epochs}, Loss: {running_loss/len(dataloader):.4f}')

10/100, Loss: 0.0102
20/100, Loss: 0.0004
30/100, Loss: 0.0002
40/100, Loss: 0.0001
50/100, Loss: 0.0001
60/100, Loss: 0.0000
70/100, Loss: 0.0000
80/100, Loss: 0.0000
90/100, Loss: 0.0000
100/100, Loss: 0.0000


In [201]:
outputs.size()

torch.Size([20, 2])

In [200]:
labels.size()

torch.Size([20])